# 03 — Gold: fact_invoice

| Property | Value |
|----------|-------|
| **Gold Table** | `fact_invoice` |
| **Grain** | One row per DocumentKey (invoice document) |
| **Source** | `rpt.vwCFInvoice` LEFT JOIN `rpt.vwTransaction` (for dimension FKs) |
| **PK** | `DocumentKey` (string) |
| **Rows** | ~9,744,807 |

### What this notebook does:
1. Read `vwCFInvoice` (9.7M rows)
2. Read `vwTransaction` to get dimension FKs (PolicyId, ProductId, Geography, Segment)
3. LEFT JOIN invoice to transaction on TransactionId
4. Add DateKey columns
5. DQ checks
6. Write to gold

> ⚠️ **Coverage warning**: Only 3 of 56 data sources contribute invoices.
> This covers ~9.5% of transactions.
>
> **Enrichment**: vwCFInvoice only has TransactionId and PartyId.
> We join to vwTransaction to inherit PolicyId, ProductId, Geography, and Segment FKs.

In [ ]:
# ============================================================
# Cell 1: Setup & Config
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark.conf.set("spark.sql.parquet.datetimeRebaseModeInRead", "CORRECTED")
spark.conf.set("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED")

LAKEHOUSE = "The_Global_Loom"
TABLE = "fact_invoice"
SOURCE_INVOICE = "rpt.vwCFInvoice"
SOURCE_TRANSACTION = "rpt.vwTransaction"

print(f"Config: {SOURCE_INVOICE} + {SOURCE_TRANSACTION} → {LAKEHOUSE}.{TABLE}")

## Cell 2: Read sources

In [ ]:
# ============================================================
# Cell 2: Read sources
# ============================================================
df_invoice = spark.table(SOURCE_INVOICE)
df_transaction = spark.table(SOURCE_TRANSACTION)

print(f"Invoices:     {df_invoice.count():,} rows × {len(df_invoice.columns)} cols")
print(f"Transactions: {df_transaction.count():,} rows × {len(df_transaction.columns)} cols")

## Cell 3: Prepare transaction dimension FKs for enrichment

Select only the columns we need from vwTransaction to enrich the invoice.

In [ ]:
# ============================================================
# Cell 3: Prepare transaction dimension FKs
# ============================================================

# OPTIMIZED: Removed denormalized columns (SegmentCode, GlobalProductClass, GlobalProductLine)
# These are already in dimension tables, reducing join overhead and result size

df_txn_fks = df_transaction.select(
    F.col("TransactionId").cast("int"),
    F.col("PolicyId").cast("int"),
    F.col("ProductId").cast("int"),
    F.col("GlobalFinancialGeographyId").cast("int"),
    F.col("GlobalFinancialSegmentId").cast("int"),
    F.col("GlobalLegalEntityId").cast("int")
)

print(f"Transaction FK lookup: {df_txn_fks.count():,} rows × {len(df_txn_fks.columns)} cols (optimized from 9 to 6)")


## Cell 4: Select invoice columns and JOIN with transaction FKs

LEFT JOIN because some invoices may not match a transaction (data quality edge case).

In [ ]:
# ============================================================
# Cell 4: Select invoice columns + JOIN transaction FKs
# ============================================================

# OPTIMIZED: Removed 7 bloat columns for Direct Lake performance:
#   - DocumentNumber (high cardinality string)
#   - Company, CompanyCode (denormalized from dim_legal_entity)
#   - CustomerNumber (high cardinality string)
#   - GlobalPartyRole (should be in dim_party/bridge)
# Result: 26 → 13 columns, ~50% reduction for 9.7M rows

df_inv_clean = df_invoice.select(
    # --- Keys ---
    F.col("DocumentKey").cast("string"),
    F.col("TransactionId").cast("int"),
    F.col("DataSourceInstanceId").cast("int"),
    F.col("PartyId").cast("int"),

    # --- Classification ---
    F.col("DocumentType").cast("string"),
    F.col("Currency").cast("string"),

    # --- Dates ---
    F.col("DocumentDate").cast("timestamp"),
    F.col("DocumentDueDate").cast("timestamp"),

    # --- DateKeys (YYYYMMDD integers) ---
    F.date_format(F.col("DocumentDate"), "yyyyMMdd").cast("int").alias("DocumentDateKey"),
    F.date_format(F.col("DocumentDueDate"), "yyyyMMdd").cast("int").alias("DocumentDueDateKey"),

    # --- Flags ---
    F.col("DirectSettlement").cast("boolean"),

    # --- Financial measures ---
    F.col("DocumentAmount").cast("decimal(18,4)"),
    F.col("CorporateAmount").cast("decimal(18,4)")
)

print(f"Invoice columns selected: {df_inv_clean.count():,} rows × {len(df_inv_clean.columns)} cols (optimized from 20 to 13)")

# LEFT JOIN with transaction FKs to enrich
df_enriched = (
    df_inv_clean
    .join(
        df_txn_fks,
        on="TransactionId",
        how="left"
    )
)

enriched_count = df_enriched.count()
matched = df_enriched.filter(F.col("PolicyId").isNotNull()).count()
unmatched = enriched_count - matched

print(f"\nAfter enrichment JOIN: {enriched_count:,} rows × {len(df_enriched.columns)} cols")
print(f"   Matched to transaction: {matched:,} ({matched*100/enriched_count:.1f}%)")
print(f"   Unmatched (no txn):     {unmatched:,} ({unmatched*100/enriched_count:.1f}%)")


## Cell 5: Data quality checks

In [ ]:
# ============================================================
# Cell 5: Data quality checks
# ============================================================
total = df_enriched.count()
dupes = total - df_enriched.select("DocumentKey").distinct().count()
null_pk = df_enriched.filter(F.col("DocumentKey").isNull()).count()
null_txn = df_enriched.filter(F.col("TransactionId").isNull()).count()
null_party = df_enriched.filter(F.col("PartyId").isNull()).count()

# DSI coverage
dsi_count = df_enriched.select("DataSourceInstanceId").distinct().count()

# Enrichment coverage
has_policy = df_enriched.filter(F.col("PolicyId").isNotNull()).count()
has_product = df_enriched.filter(F.col("ProductId").isNotNull()).count()

# Financial totals
financial_totals = df_enriched.agg(
    F.sum("DocumentAmount").alias("TotalDocumentAmount"),
    F.sum("CorporateAmount").alias("TotalCorporateAmount")
).collect()[0]

print(f"DQ Checks")
print(f"   Total rows:              {total:,}")
print(f"   Duplicate PKs:           {dupes}")
print(f"   Null DocumentKeys:       {null_pk}")
print(f"   Null TransactionIds:     {null_txn:,}")
print(f"   Null PartyIds:           {null_party:,}")
print(f"")
print(f"   Coverage:")
print(f"     Data sources:          {dsi_count} (expected ~3 of 56)")
print(f"     Has PolicyId:          {has_policy:,} ({has_policy*100/total:.1f}%)")
print(f"     Has ProductId:         {has_product:,} ({has_product*100/total:.1f}%)")
print(f"")
print(f"   Financial Totals:")
print(f"     DocumentAmount:   {financial_totals['TotalDocumentAmount']:,.2f}")
print(f"     CorporateAmount:  {financial_totals['TotalCorporateAmount']:,.2f}")

assert dupes == 0, f"ERROR: Found {dupes} duplicate DocumentKeys!"
assert null_pk == 0, f"ERROR: Found {null_pk} null DocumentKeys!"
print("\nAll DQ checks passed")

## Cell 6: Write to gold lakehouse

In [ ]:
# ============================================================
# Cell 6: Write to gold lakehouse
# ============================================================
df_enriched.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE)

print(f"Written: {TABLE}")
print(f"   Rows: {spark.table(TABLE).count():,}")